# Кейс 3. Шаг 4: гипотезы по данным БД

Один запуск на все гипотезы, для которых не нужны клиентские логи: уточнение H3, H1 (мультиаккаунты ради триала), H2 (реклама для расширения лимита, часть по БД), H5 (неуспешные платежи), H6 (длинные периоды), H7 (автопродление и продления), H8 (гости), H9 (нагрузка на бесплатный пул нод).

Нужны Parquet и витрина `mart_users.parquet` на Drive (ноутбуки 00 и 02). Каждый раздел выполняется независимо: ошибка в одном пишется в отчёт и не останавливает остальные. Итог: `h_db_results.txt` и графики в `case3/fig/`.

Время везде в UTC (поправка для `users`, логов подключений и рекламы сделана так же, как в ноутбуке 02).

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/case3"
import os, traceback, numpy as np, pandas as pd, duckdb, matplotlib.pyplot as plt
from math import sqrt
from google.colab import drive
drive.mount("/content/drive")
PARQUET, FIG = f"{DRIVE_DIR}/parquet", f"{DRIVE_DIR}/fig"
os.makedirs(FIG, exist_ok=True)
con = duckdb.connect()
con.execute("SET preserve_insertion_order = false")
for f in sorted(os.listdir(PARQUET)):
    if f.endswith(".parquet") and not f.startswith("user_connection_logs_"):
        con.execute(f"CREATE OR REPLACE VIEW {f[:-8]} AS SELECT * FROM '{PARQUET}/{f}'")
con.execute("CREATE OR REPLACE VIEW mart AS SELECT * FROM mart_users")
con.execute(f"""CREATE OR REPLACE VIEW conn_u AS SELECT * REPLACE (created_at - INTERVAL 3 HOUR AS created_at)
    FROM read_parquet('{PARQUET}/user_connection_logs_*.parquet')""")
con.execute("CREATE OR REPLACE VIEW adv_u AS SELECT * REPLACE (created_at - INTERVAL 3 HOUR AS created_at) FROM adv_view_history")
DATA_END = "2026-09-29 00:00:00"

REPORT = []
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
def out(*a):
    s = " ".join(str(x) for x in a); print(s); REPORT.append(s)
def show(title, sql, n=60):
    out(f"\n## {title}")
    try:
        df = con.sql(sql).df(); out(df.head(n).to_string(index=False) if len(df) else "(пусто)"); return df
    except Exception as e:
        out(f"ОШИБКА: {e}")
def save():
    open(f"{DRIVE_DIR}/h_db_results.txt", "w", encoding="utf-8").write("\n".join(REPORT))
def section(name, fn):
    out(f"\n{'=' * 70}\n# {name}")
    try:
        fn()
    except Exception:
        out("ОШИБКА РАЗДЕЛА:\n" + traceback.format_exc(limit=3))
    save()

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (round(float(c - h), 4), round(float(c + h), 4))

def ztest(k1, n1, k2, n2):
    """Двусторонний z-тест разности долей; возвращает (разница, p-value)."""
    from scipy.stats import norm
    p = (k1 + k2) / (n1 + n2); se = sqrt(p * (1 - p) * (1 / n1 + 1 / n2))
    if se == 0: return (0.0, 1.0)
    z = (k1 / n1 - k2 / n2) / se
    return (round(k1 / n1 - k2 / n2, 4), float(2 * (1 - norm.cdf(abs(z)))))

C = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "yellow": "#eda100", "violet": "#6250d6",
     "ink": "#0b0b0b", "ink2": "#52514e", "grid": "#e4e3df"}
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 200, "axes.edgecolor": C["ink2"], "axes.labelcolor": C["ink"],
    "xtick.color": C["ink2"], "ytick.color": C["ink2"], "axes.grid": True, "grid.color": C["grid"], "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "axes.titlesize": 11, "lines.linewidth": 2})
pct = plt.FuncFormatter(lambda v, _: f"{v:.0%}")

# Заказы с типом: триал / первая покупка / повторная (есть оплаченная обычная раньше)
con.execute("""CREATE OR REPLACE TABLE ord AS
    SELECT *, try_cast(amount AS DOUBLE) AS amt,
           CASE WHEN trial = 't' THEN 'trial'
                WHEN coalesce(sum(CASE WHEN trial = 'f' AND status = 'paid' THEN 1 ELSE 0 END)
                     OVER (PARTITION BY user_id ORDER BY created_at ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING), 0) = 0
                THEN 'first' ELSE 'repeat' END AS order_type
    FROM orders""")
print(con.sql("SELECT order_type, status, count(*) n FROM ord GROUP BY ALL ORDER BY 1, 2").df().to_string(index=False))

## H3 (уточнение): почему с мая 2026 выросла доля покупок без триала и правда ли покупка после триала — автосписание

In [ ]:
def h3_extra():
    show("признак автопродления у первой покупки, по пути к ней", f"""
        SELECT CASE WHEN first_trial_paid < first_regular_paid THEN 'триал → покупка' ELSE 'покупка без триала' END AS path,
               first_paid_pre_payment, count(*) n
        FROM mart WHERE role = 'client' AND first_regular_paid IS NOT NULL AND reg_at >= '2025-06-10'
        GROUP BY ALL ORDER BY 1, 2""")
    show("покупка через 2,5–3,5 дня после триала: признак автопродления и сумма", """
        SELECT o.pre_payment, o.amt, count(*) n FROM mart m JOIN ord o ON o.user_id = m.user_id AND o.created_at = m.first_regular_paid
        WHERE m.first_trial_paid IS NOT NULL AND epoch(m.first_regular_paid - m.first_trial_paid) BETWEEN 2.5 * 86400 AND 3.5 * 86400
        GROUP BY ALL ORDER BY n DESC LIMIT 10""")
    show("покупки без триала по месяцу регистрации: бывший гость, оплата в первый час, первая попытка триала была неуспешной", """
        SELECT reg_month, count(*) n,
               round(avg(was_guest::INT), 3) was_guest,
               round(avg((epoch(first_regular_paid - reg_at) < 3600)::INT), 3) paid_within_1h,
               round(avg((first_trial_attempt IS NOT NULL)::INT), 3) had_failed_trial_attempt,
               round(avg((platform = 'android')::INT), 3) android
        FROM mart WHERE role = 'client' AND first_regular_paid IS NOT NULL AND reg_at >= '2025-06-10'
              AND (first_trial_paid IS NULL OR first_trial_paid > first_regular_paid)
        GROUP BY 1 ORDER BY 1""")
    show("число оплаченных триалов и регистраций клиентов по месяцам (доступность триала)", """
        SELECT strftime(reg_at, '%Y-%m') m, count(*) clients,
               count(first_trial_attempt) trial_attempts, count(first_trial_paid) trial_paid,
               round(count(first_trial_paid) / count(*), 4) trial_rate
        FROM mart WHERE role = 'client' AND reg_at >= '2025-06-10' GROUP BY 1 ORDER BY 1""")
section("H3 уточнение", h3_extra)

## H1. Мультиаккаунты ради триала

**Формулировка.** Часть пользователей заводит новые аккаунты на том же устройстве или с той же картой, чтобы снова взять триал. Такие «повторные» триалы реже конвертируются в покупку.

**Метод.** Аккаунты объединяются в группы, если у них общий идентификатор устройства (`devices.device_identifier`) или общий отпечаток карты (хеш BIN + последних 4 цифр + срока). Внутри группы триалы упорядочиваются по времени: первый считается «первичным», остальные «повторными». Конверсия в покупку за 30 дней сравнивается с триалами одиночных аккаунтов. Общий IP проверяется отдельно как слабый признак (мобильные операторы раздают один IP многим абонентам).

In [ ]:
def h1():
    edges = []
    for src, sql in [("device", """SELECT list(DISTINCT user_id) u FROM devices
                        WHERE device_identifier IS NOT NULL AND user_id IS NOT NULL
                        GROUP BY device_identifier HAVING count(DISTINCT user_id) BETWEEN 2 AND 50"""),
                     ("card", """SELECT list(DISTINCT user_id) u FROM saved_payment_methods
                        WHERE card_fp IS NOT NULL GROUP BY card_fp HAVING count(DISTINCT user_id) BETWEEN 2 AND 50""")]:
        groups = con.sql(sql).df()["u"]
        out(f"групп по признаку «{src}» (2–50 аккаунтов): {len(groups)}")
        edges += [(src, list(g)) for g in groups]
    out("(группы больше 50 аккаунтов отброшены как технические идентификаторы)")

    parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    src_of = {}
    for src, users in edges:
        r0 = find(users[0])
        for u in users[1:]:
            r = find(u)
            if r != r0: parent[r] = r0
        for u in users:
            src_of.setdefault(u, set()).add(src)
    cl = pd.DataFrame({"user_id": list(parent)})
    cl["cluster"] = cl["user_id"].map(find)
    cl["src"] = cl["user_id"].map(lambda u: "+".join(sorted(src_of.get(u, []))))
    con.register("cl_df", cl)

    T = con.sql(f"""
        SELECT m.user_id, m.reg_at, m.first_trial_paid t, m.first_regular_paid p, m.platform, c.cluster, c.src,
               m.first_conn, m.last_conn
        FROM mart m LEFT JOIN cl_df c USING (user_id) WHERE m.first_trial_paid IS NOT NULL""").df()
    T["in_cluster"] = T["cluster"].notna()
    T = T.sort_values("t")
    T["rank_in_cluster"] = T.groupby("cluster").cumcount().where(T["in_cluster"]) + 1
    T["trials_in_cluster"] = T.groupby("cluster")["user_id"].transform("count").where(T["in_cluster"])
    T["group"] = np.select([~T["in_cluster"] | (T["trials_in_cluster"] == 1), T["rank_in_cluster"] == 1],
                           ["одиночный триал", "первый триал в группе"], default="повторный триал в группе")
    out(f"\nаккаунтов в группах: {len(cl)}, групп: {cl['cluster'].nunique()}")
    out("размер групп:", cl.groupby("cluster").size().value_counts().sort_index().head(10).to_dict())
    out("признак связи у аккаунтов в группах:", cl["src"].value_counts().to_dict())
    multi = T[T["trials_in_cluster"] >= 2]
    out(f"оплаченных триалов всего: {len(T)}; групп с ≥2 триалами: {multi['cluster'].nunique()}; "
        f"повторных триалов: {(T['group'] == 'повторный триал в группе').sum()} "
        f"({(T['group'] == 'повторный триал в группе').mean():.2%} всех триалов)")
    out("триалов в одной группе:", multi.groupby("cluster").size().value_counts().sort_index().to_dict())
    gaps = multi.groupby("cluster")["t"].apply(lambda s: s.sort_values().diff().dt.total_seconds().div(86400).dropna()).reset_index(drop=True)
    out("дней между соседними триалами в группе, квантили 25/50/75:", np.round(gaps.quantile([.25, .5, .75]).values, 1).tolist())

    end = pd.Timestamp(DATA_END)
    mature = T[T["t"] <= end - pd.Timedelta(days=30)].copy()
    mature["conv30"] = ((mature["p"] - mature["t"]).dt.total_seconds().between(0, 30 * 86400))
    mature["reg_to_trial_1h"] = (mature["t"] - mature["reg_at"]).dt.total_seconds() < 3600
    mature["only_trial_window"] = mature["p"].isna() & (mature["last_conn"] <= mature["t"] + pd.Timedelta(days=4))
    g = mature.groupby("group").agg(n=("conv30", "size"), conv=("conv30", "sum"),
                                    reg_to_trial_1h=("reg_to_trial_1h", "mean"), only_trial_window=("only_trial_window", "mean"))
    g["conv30"] = (g["conv"] / g["n"]).round(4)
    g["ci95"] = [wilson(k, n) for k, n in zip(g["conv"], g["n"])]
    out("\nконверсия триала в покупку за 30 дней по группам (триалы старше 30 дней):")
    out(g.round(4).to_string())
    out("only_trial_window — нет покупки и последнее подключение не позже конца триала (+1 день); "
        "reg_to_trial_1h — триал в первый час после регистрации")
    if "повторный триал в группе" in g.index:
        d, pv = ztest(g.loc["повторный триал в группе", "conv"], g.loc["повторный триал в группе", "n"],
                      g.loc["одиночный триал", "conv"], g.loc["одиночный триал", "n"])
        out(f"повторный vs одиночный: разница {d:+.4f}, p-value {pv:.2g}")
    out("\nповторные триалы по платформе:")
    out(T.groupby("platform")["group"].apply(lambda s: (s == "повторный триал в группе").mean()).round(4).sort_values(ascending=False).to_string())

    show("слабый признак: общий IP у триальщиков (только IP, на которых ≤ 5 аккаунтов всего, чтобы отсечь мобильные сети)", """
        WITH tu AS (SELECT user_id FROM mart WHERE first_trial_paid IS NOT NULL),
             ip AS (SELECT ip_address, count(DISTINCT user_id) n_all,
                           count(DISTINCT user_id) FILTER (WHERE user_id IN (SELECT user_id FROM tu)) n_trial
                    FROM sessions WHERE ip_address IS NOT NULL GROUP BY 1)
        SELECT count(*) FILTER (WHERE n_trial >= 2 AND n_all <= 5) ips_with_2plus_trials,
               sum(n_trial) FILTER (WHERE n_trial >= 2 AND n_all <= 5) trial_users_on_them,
               count(*) FILTER (WHERE n_trial >= 2) ips_with_2plus_trials_any_size
        FROM ip""")

    fig, ax = plt.subplots(figsize=(6.4, 2.2))
    order = ["одиночный триал", "первый триал в группе", "повторный триал в группе"]
    gg = g.reindex(order).dropna(subset=["n"])
    cols = [C["blue"], C["aqua"], C["orange"]][:len(gg)]
    ax.barh(gg.index[::-1], gg["conv30"].values[::-1], color=cols[::-1], height=0.55)
    for y, (v, ci, n) in enumerate(zip(gg["conv30"].values[::-1], gg["ci95"].values[::-1], gg["n"].values[::-1])):
        ax.errorbar(v, y, xerr=[[v - ci[0]], [ci[1] - v]], color=C["ink2"], capsize=3, linewidth=1)
        ax.text(ci[1] + 0.015, y, f"{v:.0%} (n = {int(n)})", va="center", color=C["ink"])
    ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(pct); ax.grid(axis="y", visible=False)
    ax.set_title("Конверсия триала в покупку за 30 дней", loc="left")
    fig.tight_layout(); fig.savefig(f"{FIG}/h1_conversion.png"); plt.show()
section("H1 Мультиаккаунты ради триала", h1)

## H2 (часть по БД). Реклама для расширения лимита

**Формулировка.** Лимит бесплатного тарифа упирается в реальное потребление у заметной доли бесплатных пользователей: они регулярно смотрят рекламу, чтобы продлить лимит. Активные зрители рекламы не чаще покупают подписку (реклама заменяет покупку).

Реальное потребление в мегабайтах есть только в клиентских логах (ноутбук 04). Здесь: частота просмотров на день бесплатного использования и связь просмотров с покупкой. Для связи с покупкой берём клиентов, не купивших в первые 7 дней, и смотрим покупку в дни 8–37 в зависимости от числа просмотров в первые 7 дней (чтобы покупка не влияла на просмотры в обратную сторону).

In [ ]:
def h2():
    show("дни бесплатного использования и реклама по месяцам", """
        WITH f AS (SELECT DISTINCT user_id, CAST(created_at AS DATE) d FROM conn_u WHERE subscribe_id IS NULL),
             a AS (SELECT user_id, CAST(created_at AS DATE) d, count(*) n FROM adv_u GROUP BY 1, 2)
        SELECT strftime(f.d, '%Y-%m') m, count(*) free_user_days,
               round(avg((a.n IS NOT NULL)::INT), 4) share_days_with_ad,
               round(avg(coalesce(a.n, 0)), 3) views_per_free_day,
               quantile_cont(a.n, 0.5) median_views_if_any,
               round(avg((coalesce(a.n, 0) >= 5)::INT), 4) share_days_5plus,
               round(avg((coalesce(a.n, 0) >= 10)::INT), 4) share_days_10plus
        FROM f LEFT JOIN a USING (user_id, d) GROUP BY 1 ORDER BY 1""")
    show("реклама по платформам (бесплатные дни с февраля 2026)", """
        WITH f AS (SELECT DISTINCT user_id, CAST(created_at AS DATE) d FROM conn_u WHERE subscribe_id IS NULL),
             a AS (SELECT user_id, CAST(created_at AS DATE) d, count(*) n FROM adv_u GROUP BY 1, 2)
        SELECT m.platform, m.role, count(*) free_user_days, round(avg((a.n IS NOT NULL)::INT), 4) share_days_with_ad
        FROM f JOIN mart m ON m.user_id = f.user_id LEFT JOIN a ON a.user_id = f.user_id AND a.d = f.d
        GROUP BY 1, 2 HAVING count(*) > 5000 ORDER BY free_user_days DESC""")
    show("пользователи по числу дней с рекламой (с февраля 2026)", """
        SELECT CASE WHEN adv_days IS NULL THEN '0' WHEN adv_days = 1 THEN '1' WHEN adv_days <= 3 THEN '2–3'
                    WHEN adv_days <= 7 THEN '4–7' WHEN adv_days <= 30 THEN '8–30' ELSE '> 30' END AS adv_days_bucket,
               count(*) users, round(avg(free_conn_days), 1) avg_free_days
        FROM mart WHERE free_conn_days > 0 GROUP BY 1 ORDER BY min(coalesce(adv_days, 0))""")
    d = show("покупка в дни 8–37 после регистрации по числу просмотров рекламы в первые 7 дней (клиенты, рег. 01.02–21.08.2026, не купившие в первые 7 дней, с бесплатным использованием)", """
        WITH base AS (
            SELECT m.user_id, m.reg_at, m.first_regular_paid, m.platform
            FROM mart m
            WHERE m.role = 'client' AND m.reg_at >= '2026-02-01' AND m.reg_at < TIMESTAMP '2026-08-22'
              AND (m.first_regular_paid IS NULL OR m.first_regular_paid >= m.reg_at + INTERVAL 7 DAY)
              AND m.free_conn_days > 0),
        v AS (SELECT b.user_id, count(a.user_id) n7 FROM base b
              LEFT JOIN adv_u a ON a.user_id = b.user_id AND a.created_at < b.reg_at + INTERVAL 7 DAY
              GROUP BY 1)
        SELECT CASE WHEN n7 = 0 THEN '0' WHEN n7 <= 5 THEN '1–5' WHEN n7 <= 20 THEN '6–20' ELSE '> 20' END AS views_first_7d,
               count(*) n, sum(coalesce(b.first_regular_paid < b.reg_at + INTERVAL 37 DAY, false)::INT) paid_8_37,
               round(avg(coalesce(b.first_regular_paid < b.reg_at + INTERVAL 37 DAY, false)::INT), 4) conv
        FROM base b JOIN v USING (user_id) GROUP BY 1 ORDER BY min(n7)""")
    if d is not None and len(d):
        d["ci95"] = [wilson(k, n) for k, n in zip(d["paid_8_37"], d["n"])]
        out(d.to_string(index=False))

    m = con.sql("""
        WITH f AS (SELECT DISTINCT user_id, CAST(created_at AS DATE) d FROM conn_u WHERE subscribe_id IS NULL),
             a AS (SELECT user_id, CAST(created_at AS DATE) d, count(*) n FROM adv_u GROUP BY 1, 2)
        SELECT date_trunc('week', f.d) w, avg((a.n IS NOT NULL)::INT) share_days_with_ad
        FROM f LEFT JOIN a USING (user_id, d) GROUP BY 1 ORDER BY 1""").df()
    fig, ax = plt.subplots(figsize=(6.4, 2.8))
    ax.plot(pd.to_datetime(m["w"]), m["share_days_with_ad"], color=C["blue"])
    ax.yaxis.set_major_formatter(pct); ax.set_ylim(0, None)
    ax.set_title("Доля дней бесплатного использования с просмотром рекламы (по неделям)", loc="left")
    fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(f"{FIG}/h2_ads_share.png"); plt.show()
section("H2 Реклама и лимит (по БД)", h2)

## H5. Неуспешные платежи

**Формулировка.** Около 40% заказов не оплачиваются, основная причина — нехватка средств. Часть пользователей после неудачной первой оплаты не возвращается, это прямая потеря выручки. Отдельно: всплеск зависших платежей в мае–июне 2026.

In [ ]:
def h5():
    show("доля неуспеха по типу заказа", """
        SELECT order_type, count(*) n, round(avg((status = 'failed')::INT), 4) failed,
               round(avg((status = 'pending')::INT), 4) pending, round(avg((status = 'paid')::INT), 4) paid
        FROM ord GROUP BY 1 ORDER BY 1""")
    show("доля неуспеха: тип заказа × период (обычные заказы)", """
        SELECT order_type, period_id, count(*) n, round(avg((status = 'failed')::INT), 4) failed
        FROM ord WHERE order_type <> 'trial' GROUP BY 1, 2 HAVING count(*) > 30 ORDER BY 1, 2""")
    show("доля неуспеха по сумме (первая покупка, Yookassa, месяц)", """
        SELECT CASE WHEN amt < 200 THEN '< 200' WHEN amt < 300 THEN '200–299' WHEN amt < 500 THEN '300–499'
                    WHEN amt < 1000 THEN '500–999' ELSE '≥ 1000' END AS amount_rub,
               count(*) n, round(avg((status = 'failed')::INT), 4) failed
        FROM ord WHERE order_type = 'first' AND provider_id = '2' AND period_id = '1'
        GROUP BY 1 ORDER BY min(amt)""")
    show("причины отказа Yookassa по типу заказа", """
        SELECT o.order_type, t.yk_cancel_reason, count(*) n,
               round(count(*) / sum(count(*)) OVER (PARTITION BY o.order_type), 4) share
        FROM transactions t JOIN ord o ON o.id = t.order_id
        WHERE t.yk_cancel_reason IS NOT NULL GROUP BY 1, 2 QUALIFY row_number() OVER (PARTITION BY o.order_type ORDER BY count(*) DESC) <= 5
        ORDER BY 1, n DESC""")
    show("что происходит после неуспешного заказа (заказы старше 7 дней)", f"""
        SELECT o.order_type, count(*) failed_orders,
               round(avg(EXISTS (SELECT 1 FROM ord p WHERE p.user_id = o.user_id AND p.status = 'paid'
                     AND p.created_at > o.created_at AND p.created_at <= o.created_at + INTERVAL 1 HOUR)::INT), 4) paid_within_1h,
               round(avg(EXISTS (SELECT 1 FROM ord p WHERE p.user_id = o.user_id AND p.status = 'paid'
                     AND p.created_at > o.created_at AND p.created_at <= o.created_at + INTERVAL 7 DAY)::INT), 4) paid_within_7d,
               round(avg(NOT EXISTS (SELECT 1 FROM ord p WHERE p.user_id = o.user_id AND p.status = 'paid'
                     AND p.created_at > o.created_at)::INT), 4) never_paid_after
        FROM ord o WHERE o.status = 'failed' AND o.created_at < TIMESTAMP '{DATA_END}' - INTERVAL 7 DAY
        GROUP BY 1 ORDER BY 1""")
    show("пользователи, у которых первая покупка так и не прошла (попытки старше 7 дней)", f"""
        WITH u AS (SELECT user_id, min(created_at) first_try,
                          bool_or(status = 'paid') any_paid, count(*) tries,
                          arg_min(amt, created_at) first_amount
                   FROM ord WHERE order_type = 'first' GROUP BY 1)
        SELECT count(*) users_tried, sum((NOT any_paid)::INT) never_paid,
               round(avg((NOT any_paid)::INT), 4) share_never_paid,
               round(avg(tries) FILTER (WHERE NOT any_paid), 2) avg_tries_never_paid,
               round(sum(first_amount) FILTER (WHERE NOT any_paid)) lost_first_amount_rub_upper_bound
        FROM u WHERE first_try < TIMESTAMP '{DATA_END}' - INTERVAL 7 DAY""")
    show("всплеск pending: по неделям, Yookassa, тип заказа", """
        SELECT date_trunc('week', created_at) w, order_type, count(*) n,
               round(avg((status = 'pending')::INT), 3) pending, round(avg((status = 'failed')::INT), 3) failed
        FROM ord WHERE provider_id = '2' AND created_at >= '2026-04-20' AND created_at < '2026-07-20'
        GROUP BY 1, 2 ORDER BY 1, 2""", n=80)

    f = con.sql("""SELECT order_type, coalesce(period_id, '?') period_id, count(*) n, avg((status = 'failed')::INT) failed
                   FROM ord WHERE order_type <> 'trial' AND period_id IS NOT NULL GROUP BY 1, 2""").df()
    lab = {"1": "1 мес.", "2": "6 мес.", "3": "1 год", "4": "2 года"}
    fig, ax = plt.subplots(figsize=(6.4, 2.8))
    x = np.arange(4); w = 0.38
    for i, (t, col, name) in enumerate([("first", C["blue"], "первая покупка"), ("repeat", C["orange"], "повторная")]):
        s = f[f["order_type"] == t].set_index("period_id").reindex(["1", "2", "3", "4"])
        ax.bar(x + (i - 0.5) * w, s["failed"], width=w - 0.04, color=col, label=name)
    ax.set_xticks(x, [lab[k] for k in ["1", "2", "3", "4"]]); ax.yaxis.set_major_formatter(pct)
    ax.set_title("Доля неуспешных заказов по периоду подписки", loc="left"); ax.legend(frameon=False)
    ax.grid(axis="x", visible=False)
    fig.tight_layout(); fig.savefig(f"{FIG}/h5_failed_by_period.png"); plt.show()
section("H5 Неуспешные платежи", h5)

## H6. Длинные периоды подписки

**Формулировка.** Скидки 15–60% на 6, 12 и 24 месяца почти не работают: длинные периоды составляют доли процента продаж, а попытки их купить чаще проваливаются. Пользователи, не сумевшие оплатить длинный период, переходят на месячный.

In [ ]:
def h6():
    show("оплаченные обычные заказы по периоду", """
        SELECT period_id, count(*) n_paid, round(count(*) / sum(count(*)) OVER (), 4) share_paid,
               round(avg(amt), 1) avg_amount_rub, round(sum(amt)) revenue_rub
        FROM ord WHERE order_type <> 'trial' AND status = 'paid' AND provider_id = '2' GROUP BY 1 ORDER BY 1""")
    show("после неуспешной попытки длинного периода: оплата месячного в течение суток", """
        SELECT o.period_id, count(*) failed_long,
               round(avg(EXISTS (SELECT 1 FROM ord p WHERE p.user_id = o.user_id AND p.status = 'paid' AND p.period_id = '1'
                     AND p.created_at > o.created_at AND p.created_at <= o.created_at + INTERVAL 1 DAY)::INT), 4) paid_monthly_1d,
               round(avg(EXISTS (SELECT 1 FROM ord p WHERE p.user_id = o.user_id AND p.status = 'paid' AND p.period_id = o.period_id
                     AND p.created_at > o.created_at AND p.created_at <= o.created_at + INTERVAL 1 DAY)::INT), 4) paid_same_long_1d
        FROM ord o WHERE o.status = 'failed' AND o.period_id IN ('2', '3', '4') GROUP BY 1 ORDER BY 1""")
    show("причины отказа Yookassa: длинный период vs месяц", """
        SELECT CASE WHEN o.period_id = '1' THEN 'месяц' ELSE 'длинный' END AS period, t.yk_cancel_reason, count(*) n,
               round(count(*) / sum(count(*)) OVER (PARTITION BY CASE WHEN o.period_id = '1' THEN 'месяц' ELSE 'длинный' END), 4) share
        FROM transactions t JOIN ord o ON o.id = t.order_id
        WHERE t.yk_cancel_reason IS NOT NULL AND o.order_type <> 'trial'
        GROUP BY 1, 2 QUALIFY row_number() OVER (PARTITION BY CASE WHEN o.period_id = '1' THEN 'месяц' ELSE 'длинный' END ORDER BY count(*) DESC) <= 4 ORDER BY 1, n DESC""")
    show("оплаченные длинные периоды по месяцам", """
        SELECT strftime(created_at, '%Y-%m') m, count(*) FILTER (WHERE period_id = '1') monthly,
               count(*) FILTER (WHERE period_id IN ('2', '3', '4')) long_periods,
               round(count(*) FILTER (WHERE period_id IN ('2', '3', '4')) / count(*), 4) long_share
        FROM ord WHERE order_type <> 'trial' AND status = 'paid' GROUP BY 1 ORDER BY 1""")
section("H6 Длинные периоды", h6)

## H7. Автопродление и продления

**Формулировка.** Подписчики с включённым автопродлением (`pre_payment`) продлевают подписку заметно чаще и остаются дольше. Сравнение наблюдательное: автопродление включают более мотивированные пользователи, поэтому вывод о причинности не делаем.

Продление: следующая оплаченная обычная подписка того же пользователя не позже чем через 7 дней после окончания первой. Берём первые покупки на 1 месяц, у которых окончание + 7 дней уже наступило.

In [ ]:
def h7():
    base_sql = f"""
        WITH fp AS (SELECT o.id, o.user_id, o.created_at, o.pre_payment, o.product_id, s.expired_time
                    FROM ord o JOIN subscribe s ON s.order_id = o.id
                    WHERE o.order_type = 'first' AND o.status = 'paid' AND o.period_id = '1')
        SELECT fp.*, EXISTS (SELECT 1 FROM ord p WHERE p.user_id = fp.user_id AND p.status = 'paid' AND p.trial = 'f'
                             AND p.created_at > fp.created_at + INTERVAL 1 DAY
                             AND p.created_at <= fp.expired_time + INTERVAL 7 DAY) AS renewed
        FROM fp WHERE fp.expired_time + INTERVAL 7 DAY <= TIMESTAMP '{DATA_END}'"""
    con.execute(f"CREATE OR REPLACE TABLE h7_base AS {base_sql}")
    d = show("продление первой месячной подписки: автопродление vs нет", """
        SELECT pre_payment, count(*) n, sum(renewed::INT) renewed, round(avg(renewed::INT), 4) renew_rate
        FROM h7_base GROUP BY 1 ORDER BY 1""")
    if d is not None and len(d) == 2:
        a, b = d.set_index("pre_payment").loc["t"], d.set_index("pre_payment").loc["f"]
        out("95% ДИ: с автопродлением", wilson(a["renewed"], a["n"]), "; без", wilson(b["renewed"], b["n"]))
        diff, pv = ztest(a["renewed"], a["n"], b["renewed"], b["n"])
        out(f"разница {diff:+.4f}, p-value {pv:.2g}")
    show("продление по тарифу и автопродлению", """
        SELECT o.product_id, p.name, h.pre_payment, count(*) n, round(avg(h.renewed::INT), 4) renew_rate
        FROM h7_base h JOIN product p ON p.id = h.product_id JOIN ord o ON o.id = h.id
        GROUP BY ALL HAVING count(*) > 100 ORDER BY 1, 3""")
    show("продление по пути к покупке (после триала или нет)", """
        SELECT CASE WHEN m.first_trial_paid < h.created_at THEN 'после триала' ELSE 'без триала' END AS path,
               h.pre_payment, count(*) n, round(avg(h.renewed::INT), 4) renew_rate
        FROM h7_base h JOIN mart m USING (user_id) GROUP BY ALL ORDER BY 1, 2""")
    show("оплаченных месяцев за 180 дней после первой покупки (первые покупки старше 180 дней)", f"""
        WITH fp AS (SELECT user_id, created_at, pre_payment FROM ord WHERE order_type = 'first' AND status = 'paid'
                    AND created_at <= TIMESTAMP '{DATA_END}' - INTERVAL 180 DAY)
        SELECT fp.pre_payment, count(*) users,
               round(avg((SELECT count(*) FROM ord p WHERE p.user_id = fp.user_id AND p.status = 'paid' AND p.trial = 'f'
                          AND p.created_at >= fp.created_at AND p.created_at < fp.created_at + INTERVAL 180 DAY)), 2) avg_paid_orders_180d
        FROM fp GROUP BY 1 ORDER BY 1""")
    show("отключение автопродления: подписки с pre_payment, доля отключённых до конца срока", """
        SELECT pre_payment, count(*) n, round(avg((disabled_at IS NOT NULL AND disabled_at < expired_time)::INT), 4) disabled_before_expiry
        FROM subscribe GROUP BY 1 ORDER BY 1""")
section("H7 Автопродление и продления", h7)

## H8. Гости и их переход в клиенты

**Формулировка.** Гостевой режим (с июня 2026) привёл основной прирост аудитории, но гости не могут платить; в клиентов переходит малая доля, и именно переход в клиенты предшествует покупке.

Ограничение: неизвестно, является ли `created_at` бывшего гостя датой создания гостевой учётной записи или датой регистрации; выводы о сроках перехода делаем осторожно.

In [ ]:
def h8():
    show("гости и бывшие гости по месяцу создания", """
        SELECT reg_month, count(*) FILTER (WHERE role = 'guest') guests, count(*) FILTER (WHERE was_guest) ex_guests_clients,
               count(*) FILTER (WHERE role = 'client' AND NOT was_guest) direct_clients,
               round(count(*) FILTER (WHERE was_guest) / nullif(count(*) FILTER (WHERE role = 'guest' OR was_guest), 0), 4) guest_to_client
        FROM mart WHERE reg_at >= '2026-05-01' GROUP BY 1 ORDER BY 1""")
    show("покупка у бывших гостей и у прямых клиентов (рег. июнь–август 2026)", """
        SELECT CASE WHEN was_guest THEN 'бывший гость' ELSE 'прямой клиент' END AS kind, count(*) n,
               round(avg((first_regular_paid IS NOT NULL)::INT), 4) paid_any,
               round(avg(coalesce(first_regular_paid < reg_at + INTERVAL 30 DAY, false)::INT), 4) paid_30d,
               round(avg((first_trial_paid IS NOT NULL)::INT), 4) trial_paid,
               round(avg(coalesce(epoch(first_regular_paid - reg_at) < 3600, false)::INT), 4) paid_within_1h
        FROM mart WHERE role = 'client' AND reg_at >= '2026-06-01' AND reg_at < '2026-09-01' GROUP BY 1""")
    show("активность гостей: дни бесплатных подключений и реклама", """
        SELECT CASE WHEN free_conn_days IS NULL OR free_conn_days = 0 THEN '0' WHEN free_conn_days = 1 THEN '1'
                    WHEN free_conn_days <= 7 THEN '2–7' WHEN free_conn_days <= 30 THEN '8–30' ELSE '> 30' END AS free_days,
               count(*) guests, round(avg((n_adv > 0)::INT), 4) watched_ads, round(avg(n_adv), 1) avg_views
        FROM mart WHERE role = 'guest' GROUP BY 1 ORDER BY min(coalesce(free_conn_days, 0))""")
    show("бывшие гости: активность до/после — дни бесплатных подключений и покупка", """
        SELECT CASE WHEN free_conn_days IS NULL OR free_conn_days = 0 THEN '0' WHEN free_conn_days <= 7 THEN '1–7'
                    WHEN free_conn_days <= 30 THEN '8–30' ELSE '> 30' END AS free_days,
               count(*) ex_guests, round(avg((first_regular_paid IS NOT NULL)::INT), 4) paid
        FROM mart WHERE was_guest GROUP BY 1 ORDER BY min(coalesce(free_conn_days, 0))""")
    show("гости по платформе", """
        SELECT platform, count(*) FILTER (WHERE role = 'guest') guests, count(*) FILTER (WHERE was_guest) ex_guests,
               round(count(*) FILTER (WHERE was_guest) / nullif(count(*) FILTER (WHERE role = 'guest' OR was_guest), 0), 4) conv
        FROM mart WHERE reg_at >= '2026-06-01' GROUP BY 1 ORDER BY guests DESC""")
section("H8 Гости", h8)

## H9. Нагрузка на бесплатный пул нод

**Формулировка.** Бесплатный трафик сосредоточен на нескольких нодах (`free_pool`), и при росте аудитории нагрузка на одну ноду росла быстрее, чем расширялся пул. Резкий обвал подключений 27.02.2026 касается только бесплатного пула.

In [ ]:
def h9():
    con.execute("""CREATE OR REPLACE TABLE h9_daily AS
        SELECT CAST(c.created_at AS DATE) d, coalesce(n.free_pool, 'f') = 't' AS free_node,
               count(*) conns, count(DISTINCT c.user_id) users, count(DISTINCT c.node_id) nodes
        FROM conn_u c LEFT JOIN nodes n ON n.id = c.node_id GROUP BY 1, 2""")
    show("по месяцам: бесплатный пул vs платные ноды (средние за день)", """
        SELECT strftime(d, '%Y-%m') m, free_node, round(avg(conns)) conns_per_day, round(avg(users)) users_per_day,
               round(avg(nodes), 1) active_nodes, round(avg(users / nodes)) users_per_node_per_day,
               max(users / nodes) max_users_per_node
        FROM h9_daily GROUP BY 1, 2 ORDER BY 1, 2""")
    show("доля крупнейшей бесплатной ноды по месяцам", """
        WITH x AS (SELECT strftime(c.created_at, '%Y-%m') m, c.node_id, count(*) n
                   FROM conn_u c JOIN nodes nd ON nd.id = c.node_id WHERE nd.free_pool = 't' GROUP BY 1, 2)
        SELECT m, count(*) free_nodes_used, round(max(n) / sum(n), 3) top1_share FROM x GROUP BY 1 ORDER BY 1""")
    show("обвал 27.02.2026: бесплатный пул vs платные ноды по дням", """
        SELECT d, free_node, conns, users FROM h9_daily WHERE d BETWEEN '2026-02-20' AND '2026-03-08' ORDER BY 1, 2""", n=60)
    show("бесплатные подключения с подпиской и без (проверка, что пул только для бесплатных)", """
        SELECT coalesce(n.free_pool, 'f') free_pool, (c.subscribe_id IS NULL) no_sub, count(*) n
        FROM conn_u c LEFT JOIN nodes n ON n.id = c.node_id GROUP BY 1, 2 ORDER BY 1, 2""")
    show("задачи обслуживания нод по месяцам (task_history)", """
        SELECT strftime(created_at, '%Y-%m') m, count(*) tasks, count(error_msg) errors FROM task_history GROUP BY 1 ORDER BY 1""")

    d = con.sql("SELECT d, users / nodes AS upn FROM h9_daily WHERE free_node ORDER BY d").df()
    d["d"] = pd.to_datetime(d["d"])
    fig, ax = plt.subplots(figsize=(6.4, 2.8))
    ax.plot(d["d"], d["upn"].rolling(7, min_periods=1).mean(), color=C["blue"])
    ax.set_ylim(0, None); ax.set_ylabel("пользователей на ноду в день")
    ax.set_title("Нагрузка на бесплатный пул (скользящее среднее за 7 дней)", loc="left")
    fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(f"{FIG}/h9_free_pool_load.png"); plt.show()
section("H9 Бесплатный пул нод", h9)
out(f"\nГотово. Отчёт: {DRIVE_DIR}/h_db_results.txt; графики: h1_conversion, h2_ads_share, h5_failed_by_period, h9_free_pool_load")
save()